# <center> Homework 106

In [4]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

## Task 5

да се тества ползата от transfer learning по отношение на време за обучение при задача 3 от домашно 105

модел B e трениран специално за за binary класификатора и при него нямаме transfer learning

    модела A да се запише след обучението
    да се сравни времето за до-обучение и model_B_on_A спрямо общото време за обучение на модел B
    да се сравни производителността на model_B_on_A спрямо тази на на модел B


In [2]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist

X_train_full, X_test = X_train_full / 255., X_test / 255.

In [3]:
A_train_mask = (y_train_full != 8) & (y_train_full != 9)
A_test_mask  = (y_test != 8) & (y_test != 9)

B_train_mask = ~A_train_mask
B_test_mask = ~A_test_mask

X_train_full_A, y_train_full_A = X_train_full[A_train_mask], y_train_full[A_train_mask]
X_train_full_B, y_train_full_B = X_train_full[B_train_mask], y_train_full[B_train_mask]
y_train_full_B = (y_train_full_B == 8).astype(int) 

X_test_A, y_test_A = X_test[A_test_mask], y_test[A_test_mask]
X_test_B, y_test_B = X_test[B_test_mask], y_test[B_test_mask]
y_test_B = (y_test_B == 8).astype(int)

X_train_A, y_train_A = X_train_full_A[:-5000], y_train_full_A[:-5000]
X_valid_A, y_valid_A = X_train_full_A[-5000:], y_train_full_A[-5000:]

X_train_B, y_train_B = X_train_full_B[:-5000], y_train_full_B[:-5000]
X_valid_B, y_valid_B = X_train_full_B[-5000:], y_train_full_B[-5000:]

In [5]:
model_A = tf.keras.models.load_model('model_A.keras')
model_A.evaluate(X_test_A, y_test_A)

2026-01-12 16:49:49.533815: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-01-12 16:49:50.081970: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 25088000 exceeds 10% of free system memory.


250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8759 - loss: 0.4018


[0.4018424153327942, 0.8758749961853027]

In [6]:
model_B = tf.keras.models.Sequential(model_A.layers[:-1]) 
model_B.add(tf.keras.layers.Dense(1, activation='sigmoid'))

for layer in model_B.layers[:-1]:
    layer.trainable = False

optimizer = tf.keras.optimizers.SGD(learning_rate=0.001)
model_B.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])

start = time.time()
model_B.fit(X_train_B, y_train_B, epochs=5, validation_data=(X_valid_B, y_valid_B))
end = time.time()

transfer_learning_fittime = end - start

for layer in model_B.layers:
    layer.trainable = True

optimizer = tf.keras.optimizers.SGD(learning_rate=0.0001)
model_B.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])

start = time.time()
model_B.fit(X_train_B, y_train_B, epochs=15, validation_data=(X_valid_B, y_valid_B))
end = time.time()

transfer_learning_fittime += end - start
_, transfer_learning_test_acc = model_B.evaluate(X_test_B, y_test_B)

print(f'--- Transfer Learning ---\nTotal Fit Time: {transfer_learning_fittime}; Test ACC: {transfer_learning_test_acc}')

Epoch 1/5


2026-01-12 16:56:35.807030: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 21952000 exceeds 10% of free system memory.


219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9507 - loss: 0.1815 - val_accuracy: 0.9706 - val_loss: 0.1136
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9753 - loss: 0.1024 - val_accuracy: 0.9788 - val_loss: 0.0910
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9806 - loss: 0.0858 - val_accuracy: 0.9820 - val_loss: 0.0792
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9831 - loss: 0.0762 - val_accuracy: 0.9840 - val_loss: 0.0715
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9844 - loss: 0.0697 - val_accuracy: 0.9846 - val_loss: 0.0661
Epoch 1/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9854 - loss: 0.0664 - val_accuracy: 0.9848 - val_loss: 0.0649
Epoch 2/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9856 - loss: 0.0651 - val_accuracy: 0.9850 - val_loss: 0.0637
Epoch 3/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9860 - loss: 0.0638 - val_accuracy: 0.9852 - val

In [7]:
model_B_without_transf = tf.keras.models.Sequential([
    tf.keras.layers.Input(X_train_B.shape[1:]),
    tf.keras.layers.Flatten(),
    
    tf.keras.layers.Dense(300, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(200, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(100, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

optimizer = tf.keras.optimizers.Adam()
model_B_without_transf.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
model_B_without_transf.fit(X_train_B, y_train_B, epochs=20, validation_data=(X_valid_B, y_valid_B))
end = time.time()

without_transfer_fittime = end - start
_, without_transfer_test_acc = model_B_without_transf.evaluate(X_test_B, y_test_B)

print(f'--- Without Transfer Learning ---\nFit Time: {without_transfer_fittime}; Test ACC: {without_transfer_test_acc}')

Epoch 1/20


2026-01-12 17:01:26.015317: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 21952000 exceeds 10% of free system memory.


219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.9923 - loss: 0.0234 - val_accuracy: 0.9972 - val_loss: 0.0093
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9980 - loss: 0.0061 - val_accuracy: 0.9976 - val_loss: 0.0088
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9986 - loss: 0.0052 - val_accuracy: 0.9982 - val_loss: 0.0071
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9994 - loss: 0.0028 - val_accuracy: 0.9988 - val_loss: 0.0086
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9980 - loss: 0.0065 - val_accuracy: 0.9988 - val_loss: 0.0054
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9990 - loss: 0.0031 - val_accuracy: 0.9990 - val_loss: 0.0055
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 1.0000 - loss: 2.2996e-04 - val_accuracy: 0.9972 - val_loss: 0.0112
Epoch 8/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9984 - loss: 0.0077 - val_accuracy: 0

In [8]:
task5_res = pd.DataFrame([
    [transfer_learning_fittime / 60, without_transfer_fittime / 60],
    [transfer_learning_test_acc, without_transfer_test_acc]
], ['Fit Time (min)', 'Test ACC'], ['Tranfer', 'Without Tranfer'])

task5_res

,Tranfer,Without Tranfer
Fit Time (min),0.979763,1.28111
Test ACC,0.987500,0.99750


## Task 6

да се тества класификатор на fasion MINST със стандартен SGD и със всички други разновидности.

    най добрия модел да се тества със batch normalization
    да се сравнят времената на обучение и производителността на обучените модели


In [11]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [10]:
optimizers = {
    'SGD': tf.keras.optimizers.SGD(),
    'SGD_momentum': tf.keras.optimizers.SGD(momentum=0.9),
    'SGD_momentum_nesterov': tf.keras.optimizers.SGD(momentum=0.9, nesterov=True),
    'Adagrad': tf.keras.optimizers.Adagrad(),
    'RMSprop': tf.keras.optimizers.RMSprop(),
    'Adam': tf.keras.optimizers.Adam(),
    'Adamax': tf.keras.optimizers.Adamax(),
    'Nadam': tf.keras.optimizers.Nadam(),
    'AdamW': tf.keras.optimizers.AdamW(),
}

from collections import defaultdict
data = defaultdict(dict) 

In [13]:
model_task6 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(300, activation="relu", kernel_initializer='he_normal'),
    tf.keras.layers.Dense(100, activation="relu", kernel_initializer='he_normal'),
    tf.keras.layers.Dense(10, activation="softmax")
])

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
for name, optimizer in optimizers.items():
    print(f'------- {name} -------')

    model = tf.keras.models.clone_model(model_task6)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

    start = time.time()
    history = model.fit(X_train, y_train, epochs=30, 
                        callbacks=callbacks, validation_data=(X_valid, y_valid))
    fit_time = time.time() - start

    test_loss, test_acc = model.evaluate(X_test, y_test)

    data[name]["Epochs"] = len(history.history["loss"])
    data[name]["Best Val Loss"] = min(history.history["val_loss"])
    data[name]["Test Loss"] = test_loss
    data[name]["Test ACC"] = test_acc
    data[name]["Fit Time"] = fit_time

------- SGD -------
Epoch 1/30


2026-01-12 17:55:11.419897: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 172480000 exceeds 10% of free system memory.


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.7663 - loss: 0.6967 - val_accuracy: 0.8086 - val_loss: 0.5466
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8339 - loss: 0.4810 - val_accuracy: 0.8378 - val_loss: 0.4652
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8474 - loss: 0.4369 - val_accuracy: 0.8498 - val_loss: 0.4235
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8576 - loss: 0.4084 - val_accuracy: 0.8604 - val_loss: 0.4069
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8645 - loss: 0.3885 - val_accuracy: 0.8558 - val_loss: 0.3977
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8690 - loss: 0.3737 - val_accuracy: 0.8624 - val_loss: 0.3886
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8742 - loss: 0.3607 - val_accuracy: 0.8642 - val_loss: 0.3673
Epoch 8/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.8793 - loss: 0.3471 - val_ac

2026-01-12 17:59:28.476613: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 31360000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8788 - loss: 0.3423
------- SGD_momentum -------
Epoch 1/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8124 - loss: 0.5241 - val_accuracy: 0.8528 - val_loss: 0.4059
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8614 - loss: 0.3824 - val_accuracy: 0.8646 - val_loss: 0.3720
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.8737 - loss: 0.3428 - val_accuracy: 0.8596 - val_loss: 0.3770
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8824 - loss: 0.3193 - val_accuracy: 0.8698 - val_loss: 0.3572
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8879 - loss: 0.3001 - val_accuracy: 0.8718 - val_loss: 0.3488
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8940 - loss: 0.2853 - val_accuracy: 0.8834 - val_loss: 0.3162
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.8992 - loss: 0.2696 - val_accuracy: 0.879

In [22]:
res_task6 = pd.DataFrame(data)
res_task6.loc['Fit Time (min)'] = res_task6.loc['Fit Time'] / 60
res_task6.drop(index='Fit Time', inplace=True)
res_task6.T.sort_values('Test ACC', ascending=False)

,Epochs,Best Val Loss,Test Loss,Test ACC,Fit Time (min)
Adamax,18.0,0.300604,0.319125,0.8903,7.733704
Nadam,12.0,0.322348,0.335495,0.8862,5.569909
Adam,12.0,0.317705,0.343454,0.8810,5.597603
SGD,23.0,0.319490,0.342346,0.8788,4.293710
AdamW,10.0,0.322893,0.344814,0.8772,4.452555
SGD_momentum_nesterov,10.0,0.312984,0.339943,0.8749,2.692260
SGD_momentum,11.0,0.316192,0.341728,0.8747,2.473622
RMSprop,8.0,0.371220,0.396185,0.8643,2.854045
Adagrad,30.0,0.386580,0.418235,0.8520,8.061788


In [25]:
best_model_task6 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),

    tf.keras.layers.Dense(300, kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Dense(100, kernel_initializer='he_normal'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),

    tf.keras.layers.Dense(10, activation="softmax")
])

best_model_task6.compile(optimizer=tf.keras.optimizers.Adamax(), 
                         loss="sparse_categorical_crossentropy", metrics=["accuracy"])
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

start = time.time()
history = best_model_task6.fit(X_train, y_train, epochs=30, 
                    callbacks=callbacks, validation_data=(X_valid, y_valid))
fit_time = time.time() - start

test_loss, test_acc = best_model_task6.evaluate(X_test, y_test)

print(f'--- Best Model Optimizer: Adamax ---\nFit Time: {fit_time / 60}; Test ACC: {test_acc}; Test Loss: {test_loss}')

Epoch 1/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 23s 12ms/step - accuracy: 0.8338 - loss: 0.4826 - val_accuracy: 0.8696 - val_loss: 0.3684
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.8729 - loss: 0.3544 - val_accuracy: 0.8708 - val_loss: 0.3586
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.8876 - loss: 0.3112 - val_accuracy: 0.8824 - val_loss: 0.3215
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.8971 - loss: 0.2823 - val_accuracy: 0.8834 - val_loss: 0.3167
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9062 - loss: 0.2611 - val_accuracy: 0.8782 - val_loss: 0.3355
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9099 - loss: 0.2450 - val_accuracy: 0.8812 - val_loss: 0.3247
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.9152 - loss: 0.2298 - val_accuracy: 0.8730 - val_loss: 0.3363
Epoch 8/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.9211 -